# Combining the four runs into the paper tables


In [ ]:
import os, json
import numpy as np
import pandas as pd

INPUT_ROOT = '/kaggle/input'
OUT_DIR = '/kaggle/working'

runs = {}
for root, _, files in os.walk(INPUT_ROOT):
    if 'metrics.json' in files and os.path.basename(root).startswith('results_'):
        with open(os.path.join(root, 'metrics.json')) as fh:
            m = json.load(fh)
        runs[m['run']] = m
print('Runs found:', sorted(runs))
patient_runs = sorted(r for r in runs if r.startswith('patient'))
assert patient_runs, 'Attach at least one patient_r* output'

def mean_sd(values):
    values = np.asarray(values, dtype=float)
    return f'{values.mean():.3f} ± {values.std(ddof=1):.3f}' if len(values) > 1 else f'{values.mean():.3f}'

BACKBONES = ['resnet18+densenet121', 'resnet50', 'densenet121', 'efficientnet_b0']
lines = []

def md_table(df):
    cols = list(df.columns)
    out = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    for _, r in df.iterrows():
        out.append('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
    return '\n'.join(out)

Runs found: ['patient_r0', 'patient_r1', 'patient_r2', 'slice']


## Table I: the same pipelines under a slice-level and a patient-level split

In [ ]:
names = [('ova', 'Hybrid ResNet-18+DenseNet-121, OvA (fine-tuned)')] + [('lr_' + b, f'{b}, frozen features + LR') for b in BACKBONES]
rows = []
for key, label in names:
    row = {'model': label}
    if 'slice' in runs:
        row['slice split: slice acc'] = f"{runs['slice']['slice'][key]['acc']:.3f}"
        row['slice split: slice macro-F1'] = f"{runs['slice']['slice'][key]['macro_F1']:.3f}"
    row['patient split: slice macro-F1'] = mean_sd([runs[r]['slice'][key]['macro_F1'] for r in patient_runs])
    row['patient split: patient macro-F1'] = mean_sd([runs[r]['patient'][key]['macro_F1'] for r in patient_runs])
    row['patient split: patient BA'] = mean_sd([runs[r]['patient'][key]['macro_R'] for r in patient_runs])
    rows.append(row)
table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))
lines += ['## Table I', md_table(table1), '']

                                          model slice split: slice acc slice split: slice macro-F1 patient split: slice macro-F1 patient split: patient macro-F1 patient split: patient BA
Hybrid ResNet-18+DenseNet-121, OvA (fine-tuned)                  0.995                       0.995                 0.387 ± 0.023                   0.382 ± 0.018             0.380 ± 0.015
     resnet18+densenet121, frozen features + LR                  0.662                       0.649                 0.403 ± 0.012                   0.436 ± 0.021             0.439 ± 0.022
                 resnet50, frozen features + LR                  0.583                       0.565                 0.376 ± 0.012                   0.397 ± 0.016             0.401 ± 0.019
              densenet121, frozen features + LR                  0.657                       0.647                 0.410 ± 0.022                   0.442 ± 0.021             0.446 ± 0.025
          efficientnet_b0, frozen features + LR                  

## Main result: repeated patient-level cross-validation over all patients (mean ± SD across repeats)

In [ ]:
labels = {'ova': 'Hybrid OvA', 'two_stage': 'Two-stage (tuned on calibration set)', 'two_stage_default': 'Two-stage (hybrid stage 2, thresholds 0.5)',
          'single_selected': 'Single-stage LR (backbone chosen on calibration set)'}
labels.update({'lr_' + b: f'Single-stage LR, {b}' for b in BACKBONES})
rows = []
for key, label in labels.items():
    row = {'model': label}
    for metric in ['macro_P', 'macro_R', 'macro_F1', 'R_AD', 'R_CN', 'R_MCI', 'acc']:
        row[metric] = mean_sd([runs[r]['patient'][key][metric] for r in patient_runs])
    rows.append(row)
table2 = pd.DataFrame(rows)
print(table2.to_string(index=False))
lines += ['## Repeated CV (macro_R = balanced accuracy)', md_table(table2), '']

rows = []
for r in patient_runs:
    for comp, v in runs[r]['paired'].items():
        rows.append({'repeat': r, 'comparison': comp, 'diff': round(v['diff'], 3), 'CI': f"{v['ci'][0]:+.3f} to {v['ci'][1]:+.3f}",
                     'only A correct': v['only_a'], 'only B correct': v['only_b'], 'McNemar p': round(v['mcnemar_p'], 3)})
table3 = pd.DataFrame(rows)
print(table3.to_string(index=False))
lines += ['## Paired comparisons (A minus B, per repeat)', md_table(table3), '']

stage = pd.DataFrame([{'repeat': r, 'stage-1 AUC': round(runs[r]['stage1_AUC'], 3), 'stage-2 AUC': round(runs[r]['stage2_AUC'], 3),
                       'stage-2 BA in cascade': round(runs[r]['stage2_BA_in_cascade'], 3)} for r in patient_runs])
print(stage.to_string(index=False))
lines += ['## Stage-level results', md_table(stage), '']

                                               model       macro_P       macro_R      macro_F1          R_AD          R_CN         R_MCI           acc
                                          Hybrid OvA 0.406 ± 0.034 0.380 ± 0.015 0.382 ± 0.018 0.173 ± 0.039 0.288 ± 0.046 0.678 ± 0.031 0.485 ± 0.018
                Two-stage (tuned on calibration set) 0.415 ± 0.031 0.412 ± 0.034 0.411 ± 0.034 0.290 ± 0.102 0.341 ± 0.030 0.605 ± 0.079 0.479 ± 0.034
          Two-stage (hybrid stage 2, thresholds 0.5) 0.455 ± 0.046 0.407 ± 0.025 0.415 ± 0.030 0.204 ± 0.032 0.311 ± 0.043 0.708 ± 0.035 0.514 ± 0.024
Single-stage LR (backbone chosen on calibration set) 0.416 ± 0.023 0.418 ± 0.022 0.415 ± 0.024 0.309 ± 0.060 0.375 ± 0.080 0.569 ± 0.023 0.472 ± 0.022
               Single-stage LR, resnet18+densenet121 0.436 ± 0.018 0.439 ± 0.022 0.436 ± 0.021 0.327 ± 0.047 0.413 ± 0.063 0.577 ± 0.014 0.490 ± 0.015
                           Single-stage LR, resnet50 0.396 ± 0.015 0.401 ± 0.019 0.397 ± 0.016

## What the calibration sets chose

In [ ]:
sel = pd.DataFrame([dict(s, repeat=r) for r in patient_runs for s in runs[r]['selection']])
summary_sel = {
    'stage2_model_counts': sel['stage2_model'].value_counts().to_dict(),
    'single_backbone_counts': sel['single_backbone'].value_counts().to_dict(),
    'T1_mean_sd': [float(sel['T1'].mean()), float(sel['T1'].std())],
    'T2_mean_sd': [float(sel['T2'].mean()), float(sel['T2'].std())],
}
print(json.dumps(summary_sel, indent=1))
lines += ['## Selections across folds', '```', json.dumps(summary_sel, indent=1), '```', '']

if 'slice' in runs:
    print('Slice-split run, patient-level aggregation (patients shared between train and test):',
          round(runs['slice']['patient']['ova']['macro_F1'], 3))

with open(os.path.join(OUT_DIR, 'paper_tables.md'), 'w') as fh:
    fh.write('\n'.join(lines))
with open(os.path.join(OUT_DIR, 'paper_summary.json'), 'w') as fh:
    json.dump({'table1': table1.to_dict('records'), 'table2': table2.to_dict('records'), 'paired': table3.to_dict('records'),
               'stages': stage.to_dict('records'), 'selection': summary_sel,
               'slice_run_patient_ova': runs.get('slice', {}).get('patient', {}).get('ova')}, fh, indent=1)
print('Saved paper_tables.md and paper_summary.json')

{
 "stage2_model_counts": {
  "efficientnet_b0|rbf_svm": 3,
  "hybrid|deep": 3,
  "densenet121|logreg": 2,
  "resnet50|rbf_svm": 1,
  "densenet121|random_forest": 1,
  "resnet18+densenet121|linear_svm": 1,
  "resnet50|logreg": 1,
  "resnet18+densenet121|mlp": 1,
  "resnet18+densenet121|logreg": 1,
  "densenet121|mlp": 1
 },
 "single_backbone_counts": {
  "efficientnet_b0": 6,
  "densenet121": 4,
  "resnet18+densenet121": 3,
  "resnet50": 2
 },
 "T1_mean_sd": [
  0.32,
  0.1955578102323125
 ],
 "T2_mean_sd": [
  0.5506666666666666,
  0.13057382513170016
 ]
}
Slice-split run, patient-level aggregation (patients shared between train and test): 1.0
Saved paper_tables.md and paper_summary.json


In [ ]:
from sklearn.metrics import f1_score, confusion_matrix

dfs = []
for root, _, files in os.walk(INPUT_ROOT):
    if os.path.basename(root).startswith('results_patient') and 'patient_predictions.csv' in files:
        dfs.append(pd.read_csv(os.path.join(root, 'patient_predictions.csv')))

def mf1(y, p):
    return f1_score(y, p, labels=[0, 1, 2], average='macro', zero_division=0)

rows = []
for df in dfs:
    for k, g in df.groupby('fold'):
        for model in ['ova', 'two_stage', 'single_selected']:
            rows.append({'model': model, 'macro_F1': mf1(g['true'], g[model])})
print('Per test fold (15 folds of about 64 patients):')
print(pd.DataFrame(rows).groupby('model')['macro_F1'].agg(['min', 'max', 'mean']).round(3))

rng = np.random.default_rng(0)
for model in ['ova', 'two_stage']:
    scores = []
    for df in dfs:
        for _ in range(1000):
            g = df.groupby('true', group_keys=False).sample(frac=47 / 320, random_state=int(rng.integers(1e9)))
            scores.append(mf1(g['true'], g[model]))
    scores = np.array(scores)
    print(f'{model}, random stratified 47-patient test sets: 95th pct {np.percentile(scores, 95):.3f}, '
          f'max {scores.max():.3f}, share >= 0.60: {np.mean(scores >= 0.60):.4f}')

for model in ['ova', 'two_stage']:
    cm = sum(confusion_matrix(df['true'], df[model], labels=[0, 1, 2]) for df in dfs)
    print(model, 'confusion matrix, 3 repeats summed (rows true AD/CN/MCI, columns predicted):')
    print(cm)

Per test fold (15 folds of about 64 patients):
                   min    max   mean
model                               
ova              0.301  0.474  0.381
single_selected  0.334  0.498  0.413
two_stage        0.323  0.492  0.400
ova, random stratified 47-patient test sets: 95th pct 0.496, max 0.632, share >= 0.60: 0.0010
two_stage, random stratified 47-patient test sets: 95th pct 0.530, max 0.660, share >= 0.60: 0.0063
ova confusion matrix, 3 repeats summed (rows true AD/CN/MCI, columns predicted):
[[ 28  19 115]
 [ 19  76 169]
 [ 42 130 362]]
two_stage confusion matrix, 3 repeats summed (rows true AD/CN/MCI, columns predicted):
[[ 47  24  91]
 [ 33  90 141]
 [ 85 126 323]]
